# Mapillary API Client

**Author:** James Chukwuemeka  
**Affiliation:** University of Salford, UK

## Purpose

This notebook provides a small, reusable Python wrapper for the Mapillary Graph API. It demonstrates how to authenticate, search for public Mapillary images within a geographic bounding box, retrieve image metadata, and save the returned records as a CSV file.

## Before you begin

1. Create a Mapillary account and sign in.
2. Open the [Mapillary Developer Dashboard](https://www.mapillary.com/dashboard/developers).
3. Select **Register application**, enter the requested application details, and enable at least **Read** access for public data.
4. Find the registered application and select **View** under **Client Token**. Copy the token and keep it private.
5. Read the [Mapillary API documentation](https://www.mapillary.com/developer/api-documentation/) for available endpoints, fields and usage requirements.

The token is an access credential, not a value to share in a report or commit to a public repository. Replace the placeholder in the next section with your own token. For a shared or published notebook, use your notebook platform's secret/environment-variable facility instead of saving a real token in the file.

## 1. Install and import dependencies

In [ ]:
%pip install -q requests pandas

In [ ]:
import logging
import time
from typing import Optional, Sequence

import pandas as pd
import requests
from requests.exceptions import RequestException

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

## 2. Configure your access token

Replace the placeholder below with the Client Token from your Mapillary Developer Dashboard. Do not include quotation marks inside the token itself.

In [ ]:
MAPILLARY_ACCESS_TOKEN = "YOUR_MAPILLARY_ACCESS_TOKEN"

## 3. Define the reusable `MapillaryClient`

The class keeps authentication and HTTP request handling in one place. `search_images()` retrieves image records for a bounding box, while `get_image()` retrieves metadata for one known image ID. Transient rate-limit and server errors are retried with a short exponential delay; other HTTP errors are raised so that the caller can see what went wrong.

In [ ]:
class MapillaryClient:
    """Small client for selected Mapillary Graph API image endpoints."""

    BASE_URL = "https://graph.mapillary.com"

    DEFAULT_FIELDS = (
        "id",
        "captured_at",
        "computed_geometry",
        "compass_angle",
        "camera_type",
        "sequence",
        "thumb_1024_url",
    )

    RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

    def __init__(
        self,
        access_token: str,
        timeout: int = 30,
        max_retries: int = 3,
    ):
        if not access_token or access_token == "YOUR_MAPILLARY_ACCESS_TOKEN":
            raise ValueError("Add your Mapillary Client Token before creating the client.")

        self.session = requests.Session()
        self.session.headers.update({
            "Authorization": f"OAuth {access_token}",
            "Accept": "application/json",
        })
        self.timeout = timeout
        self.max_retries = max_retries

    def _get(self, endpoint: str, params: Optional[dict] = None) -> dict:
        """Send an authenticated GET request and return the JSON response."""
        url = f"{self.BASE_URL}/{endpoint.lstrip('/')}"

        for attempt in range(self.max_retries):
            try:
                response = self.session.get(
                    url, params=params, timeout=self.timeout
                )

                if response.status_code in self.RETRYABLE_STATUS_CODES:
                    if attempt == self.max_retries - 1:
                        response.raise_for_status()
                    wait = 2 ** attempt
                    logging.warning(
                        "Mapillary returned HTTP %s; retrying in %s seconds.",
                        response.status_code, wait
                    )
                    time.sleep(wait)
                    continue

                response.raise_for_status()
                return response.json()

            except RequestException:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 ** attempt)

        raise RuntimeError("Mapillary request failed after the configured retries.")

    def search_images(
        self,
        bbox: Sequence[float],
        fields: Optional[Sequence[str]] = None,
        limit: int = 100,
    ) -> list:
        """Search images in bbox=(west, south, east, north)."""
        if len(bbox) != 4:
            raise ValueError("bbox must contain west, south, east, north.")
        west, south, east, north = map(float, bbox)

        if not (-180 <= west <= 180 and -180 <= east <= 180):
            raise ValueError("Longitude values must be between -180 and 180.")
        if not (-90 <= south <= 90 and -90 <= north <= 90):
            raise ValueError("Latitude values must be between -90 and 90.")
        if west >= east or south >= north:
            raise ValueError("bbox must have west < east and south < north.")
        if not 1 <= int(limit) <= 2000:
            raise ValueError("limit must be between 1 and 2000.")

        requested_fields = fields or self.DEFAULT_FIELDS
        params = {
            "bbox": f"{west},{south},{east},{north}",
            "fields": ",".join(requested_fields),
            "limit": int(limit),
        }
        result = self._get("images", params=params)
        return result.get("data", [])

    def get_image(
        self,
        image_id: str,
        fields: Optional[Sequence[str]] = None,
    ) -> dict:
        """Retrieve selected metadata for a single Mapillary image ID."""
        if not image_id:
            raise ValueError("image_id must not be empty.")

        requested_fields = fields or self.DEFAULT_FIELDS
        params = {"fields": ",".join(requested_fields)}
        return self._get(str(image_id), params=params)

    @staticmethod
    def to_dataframe(records: list) -> pd.DataFrame:
        """Convert image records to a tabular form with coordinate columns."""
        rows = []
        for record in records:
            geometry = record.get("computed_geometry") or {}
            coordinates = geometry.get("coordinates") or [None, None]
            row = {
                key: value for key, value in record.items()
                if key != "computed_geometry"
            }
            row["longitude"] = coordinates[0] if len(coordinates) > 0 else None
            row["latitude"] = coordinates[1] if len(coordinates) > 1 else None
            rows.append(row)
        return pd.DataFrame(rows)

## 4. Create the client and choose example locations

The examples below use five recognisable locations in Greater Manchester. Their coordinates are approximate reference points, not guaranteed Mapillary image locations. The wrapper searches within a small bounding box around each point.

The bounding box order is **west, south, east, north** (longitude, latitude, longitude, latitude). You can replace these examples with coordinates for your own study area.

In [ ]:
client = MapillaryClient(MAPILLARY_ACCESS_TOKEN)

# Approximate central coordinates: (latitude, longitude)
locations = {
    "Manchester Museum": (53.4668, -2.2339),
    "Heaton Park": (53.5403, -2.2577),
    "The Lowry": (53.4708, -2.2970),
    "IWM North": (53.4688, -2.2985),
    "Peel Park": (53.4874, -2.2702),
}

def create_bbox(latitude, longitude, offset=0.002):
    """Create a small bbox around a point; offset is in degrees."""
    return (
        longitude - offset,  # west
        latitude - offset,   # south
        longitude + offset,  # east
        latitude + offset,   # north
    )

print(f"Mapillary client initialised. {len(locations)} example locations defined.")

## 5. Search the example locations and inspect the results

The following cell searches a small area around each example coordinate. Results are combined into one table, with an additional `search_location` column showing which example area was queried. A location may return no images; this is a valid outcome and does not necessarily indicate an error.

In [ ]:
all_image_records = []

for location_name, (latitude, longitude) in locations.items():
    bbox = create_bbox(latitude, longitude)
    records = client.search_images(bbox=bbox, limit=20)

    for record in records:
        record["search_location"] = location_name

    all_image_records.extend(records)
    print(f"{location_name}: {len(records)} images returned")

images_df = client.to_dataframe(all_image_records)

# Nearby search areas may overlap, so remove duplicate image IDs.
if not images_df.empty:
    images_df = images_df.drop_duplicates(subset="id").reset_index(drop=True)

print(f"Unique images returned across all locations: {len(images_df)}")
display(images_df.head())

## 6. Retrieve one image's metadata (optional)

Use an image ID returned by the search. This step is optional; the search response may already contain the fields needed for your analysis.

In [ ]:
if not images_df.empty:
    sample_image_id = str(images_df.iloc[0]["id"])
    image_metadata = client.get_image(sample_image_id)
    display(pd.Series(image_metadata, name="Image metadata"))
else:
    print("No images were returned. Check the bounding box and try another area.")

## 7. Save the retrieved metadata

This exports the combined metadata returned by the five example searches. It does not download the original image files.

In [ ]:
if not images_df.empty:
    output_file = "mapillary_image_metadata.csv"
    images_df.to_csv(output_file, index=False)
    print(f"Saved {len(images_df)} records to {output_file}")
else:
    print("Nothing to export.")

## Notes

- The five Greater Manchester coordinates are approximate examples. Replace them with your own coordinates when using the client for another study.
- The bounding-box offset controls the size of each search area. Nearby or overlapping boxes can return the same image; the example removes duplicate image IDs in the combined table.
- Keep your access token private. If it has been exposed, revoke or replace it through your Mapillary developer account.
- A successful API response does not guarantee that every optional field is present; inspect the returned columns before analysis.
- This notebook retrieves metadata and thumbnail URLs only. It does not download image files or integrate Mapillary imagery into the receipt preprocessing pipeline.
- Follow Mapillary's [API documentation](https://www.mapillary.com/developer/api-documentation/) and [Terms of Use](https://www.mapillary.com/terms), including any applicable attribution and image-use requirements.
- If the API returns an authentication, permission, validation or rate-limit error, check the token, application permissions, request parameters and current API guidance before retrying.

## References

- Mapillary. [API Documentation](https://www.mapillary.com/developer/api-documentation/).
- Mapillary. [Developer Dashboard](https://www.mapillary.com/dashboard/developers).
- Mapillary. [Accessing imagery and data through the Mapillary API](https://help.mapillary.com/hc/en-us/articles/360010234680-Accessing-imagery-and-data-through-the-Mapillary-API).
- Mapillary Python SDK. [Client authentication documentation](https://mapillary.github.io/mapillary-python-sdk/docs/mapillary.models/mapillary.models.client/).